In [252]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt 
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest

### We will be using isolation forest to detect the anamolies

In [253]:
data = pd.read_parquet('../updatedtop20datasets/META.parquet')
data.head(5)

,date,close,high,low,open,volume,adjClose,adjHigh,adjLow,adjOpen,adjVolume,divCash,splitFactor,Symbol,price_return,Volatility_7Days,Volatility_30Days,Price_Swing,volume_zscore7Days,volume_zscore30Days,MASignal,year,volatility_diff,volume_z_diff,price_score,volume_score,volatility_score,cum_score,is_anomaly
0,2012-05-18 00:00:00+00:00,38.2318,45.00,38.00,42.05,573576400,37.937002,44.653014,37.706990,41.725761,573576400,0.0,1.0,META,-0.043740,0.0,-0.952237,10.341857,0.0,0.006591,-1.29285,2012,0.648191,0.030607,-0.127479,0.17752,0.134807,0.061616,0
1,2012-05-21 00:00:00+00:00,34.0300,36.66,33.00,36.53,168192700,33.767602,36.377322,32.745544,36.248325,168192700,0.0,1.0,META,-4.437747,0.0,-0.952237,5.352511,0.0,0.006591,-1.29285,2012,0.648191,0.030607,-0.164785,0.17752,0.134807,0.049181,1
2,2012-05-22 00:00:00+00:00,31.0000,33.59,30.94,32.61,101786600,30.760965,33.330994,30.701428,32.358551,101786600,0.0,1.0,META,-3.603582,0.0,-0.952237,3.894950,0.0,0.006591,-1.29285,2012,0.648191,0.030607,-0.130041,0.17752,0.134807,0.060762,0
3,2012-05-23 00:00:00+00:00,32.0000,32.50,31.36,31.37,73600000,31.753255,32.249399,31.118189,31.128112,73600000,0.0,1.0,META,1.245959,0.0,-0.952237,0.601883,0.0,0.006591,-1.29285,2012,0.648191,0.030607,0.095657,0.17752,0.134807,0.135995,0
4,2012-05-24 00:00:00+00:00,33.0300,33.21,31.77,32.95,50237200,32.775312,32.953925,31.525028,32.695929,50237200,0.0,1.0,META,1.243138,0.0,-0.952237,1.128398,0.0,0.006591,-1.29285,2012,0.648191,0.030607,0.065296,0.17752,0.134807,0.125875,0


In [254]:
def data_partition(symbol):
    path = '../updatedtop20datasets/'
    filepath = f'{path}{symbol}.parquet'
    columns =  ['price_return', 'volatility_diff', 'Volatility_30Days', 'Price_Swing', 'volume_z_diff', 'volume_zscore30Days','MASignal']
    data = pd.read_parquet(filepath)
    scaler = StandardScaler()
    data[columns] = scaler.fit_transform(data[columns])
    data.to_parquet(filepath, index = False)
    return data

In [255]:
def iso_runner(symbol,contamination = 0.05):
    path = '../updatedtop20datasets/'
    filepath = f'{path}{symbol}.parquet'
    data = pd.read_parquet(filepath)
    price_features = ['price_return', 'Price_Swing', 'MASignal']
    volume_features = ['volume_z_diff', 'volume_zscore30Days']
    volatility_features = ['volatility_diff', 'Volatility_30Days']
    
    for features, col_name in zip(
        [price_features, volume_features, volatility_features],
        ['price_score', 'volume_score', 'volatility_score']
    ):
        model = IsolationForest(contamination=contamination, random_state=42)
        model.fit(data[features])
        data[col_name] = model.decision_function(data[features])
    
    data['cum_score'] = (data['price_score'] + 
                         data['volume_score'] + 
                         data['volatility_score']) / 3
    data['is_anomaly'] = (data['cum_score'] < data['cum_score'].quantile(0.05)).astype(int)
    data.to_parquet(filepath, index=False)
    return data

## For all the 20 datas we have ! 


In [256]:
### Creating datasets for the top 20 companies 
watchlist = {
    "AAPL":  "Apple",
    "MSFT":  "Microsoft", 
    "NVDA":  "Nvidia",
    "GOOGL": "Alphabet (Google)",
    "AMD":  "AMD",
    "META":  "Meta (Facebook)",
    "TSLA":  "Tesla",
    "NFLX": "Netflix",
    "LLY":   "Eli Lilly",
    "AVGO":   "Broadcom",
    "MU":    "Micron Technology",
    "QCOM":   "Qualcomm",
    "UNH":   "UnitedHealth",
    "WMT":   "Walmart",
    "MA":    "Mastercard",
    "JNJ":   "Johnson & Johnson",
    "PG":    "Procter & Gamble",
    "HD":    "Home Depot",
    "ORCL":  "Oracle",
    "JPM": "JPMorgan Chase"
}


In [257]:
michelle = data_partition("AAPL")
michelle.head(5)

,date,close,high,low,open,volume,adjClose,adjHigh,adjLow,adjOpen,adjVolume,divCash,splitFactor,Symbol,price_return,Volatility_7Days,Volatility_30Days,Price_Swing,volume_zscore7Days,volume_zscore30Days,MASignal,year,volatility_diff,volume_z_diff,price_score,volume_score,volatility_score,cum_score,is_anomaly
0,2008-01-02 00:00:00+00:00,194.84,200.26,192.55,199.27,38542100,5.832436,5.994681,5.763886,5.965046,1079179879,0.0,1.0,AAPL,-0.054074,0.0,-0.861003,1.201190,0.0,0.040594,-1.278234,2008,0.686556,-0.028592,0.115155,0.170954,0.134807,0.140305,0
1,2008-01-03 00:00:00+00:00,194.93,197.39,192.69,195.41,30073800,5.835131,5.908769,5.768077,5.849499,842067242,0.0,1.0,AAPL,-0.030299,0.0,-0.861003,0.152389,0.0,0.040594,-1.278234,2008,0.686556,-0.028592,0.174206,0.170954,0.134807,0.159989,0
2,2008-01-04 00:00:00+00:00,180.05,193.00,178.89,191.45,51994000,5.389705,5.777357,5.354981,5.730959,1455833455,0.0,1.0,AAPL,-3.983070,0.0,-0.861003,3.833159,0.0,0.040594,-1.278234,2008,0.686556,-0.028592,-0.108616,0.170954,0.134807,0.065715,0
3,2008-01-07 00:00:00+00:00,177.64,183.60,170.23,181.25,74006900,5.317563,5.495973,5.095749,5.425627,2072195272,0.0,1.0,AAPL,-0.743014,0.0,-0.861003,3.622680,0.0,0.040594,-1.278234,2008,0.686556,-0.028592,-0.035450,0.170954,0.134807,0.090104,0
4,2008-01-08 00:00:00+00:00,171.25,182.46,170.80,180.14,54422000,5.126282,5.461847,5.112811,5.392399,1523817523,0.0,1.0,AAPL,-1.905547,0.0,-0.861003,3.135786,0.0,0.040594,-1.278234,2008,0.686556,-0.028592,-0.014099,0.170954,0.134807,0.097221,0


In [258]:
for symbol in watchlist.keys():
    data_partition(symbol)
for symbol in watchlist.keys():
    iso_runner(symbol)
    print(f"{symbol} done")

AAPL done
MSFT done
NVDA done
GOOGL done
AMD done
META done
TSLA done
NFLX done
LLY done
AVGO done
MU done
QCOM done
UNH done
WMT done
MA done
JNJ done
PG done
HD done
ORCL done
JPM done


In [259]:
### Anamoly Count 
for symbol in watchlist.keys():
    data = pd.read_parquet(f'../updatedtop20datasets/{symbol}.parquet')
    counts = data['is_anomaly'].value_counts()
    print(f"{symbol}: {counts[1]/counts[0]} % anomalies")

AAPL: 0.05271529197909566 % anomalies
MSFT: 0.05271529197909566 % anomalies
NVDA: 0.05271529197909566 % anomalies
GOOGL: 0.05271529197909566 % anomalies
AMD: 0.05271529197909566 % anomalies
META: 0.052804295942720764 % anomalies
TSLA: 0.05282522996057819 % anomalies
NFLX: 0.05271529197909566 % anomalies
LLY: 0.05271529197909566 % anomalies
AVGO: 0.05274944015924359 % anomalies
MU: 0.05271529197909566 % anomalies
QCOM: 0.05271529197909566 % anomalies
UNH: 0.05271529197909566 % anomalies
WMT: 0.05271529197909566 % anomalies
MA: 0.05271529197909566 % anomalies
JNJ: 0.05271529197909566 % anomalies
PG: 0.05271529197909566 % anomalies
HD: 0.05271529197909566 % anomalies
ORCL: 0.05271529197909566 % anomalies
JPM: 0.05271529197909566 % anomalies
